# ML-09 — Validation and Research Claim Audit
Jackline Mutheu — Refresh / Content Opportunity Scoring

Working with an AI assistant: read `skills/README.md`, then load `hunting-leakage-and-validating` + `flyrank/flyrank-data` — done before writing this notebook.


## 1. Two paper findings + my methodology questions

**Finding A — "What Predicts Health?" (Random Forest feature importance, ML Appendix, p.26-27).** Average Position (43%) and Impressions (32%) are the top predictors of Health Score.

*My methodology question:* Health Score is explicitly defined in the paper's own "How to Read" section as `Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)`. Since two of the four ingredients of the label are also the model's top two "predictors," does a holdout split on the same feature-vector snapshot actually test whether the model learned something new — or does it mostly re-derive the composite formula it was given? The paper is admirably upfront that "importance is descriptive rather than causal," which is exactly the right caveat — my question is whether the finding could go one step further: reporting feature importance *with* position/impressions/CTR/scroll removed would show whether age, freshness, or word count carry any independent signal at all, or whether the whole ranking is just the label's own arithmetic reflected back.

**Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, ML Appendix, p.28).** Content Age, Days Since Update, and Days Visible are the strongest signals separating growing from declining pages.

*My methodology question:* Per the paper's own definitions page, Trend Direction (the likely source of the growth/decline label) is "calculated from 30d-vs-prev-30d impression change." If any input feature to this model overlaps with that same 30-day comparison window, that's the exact leakage pattern I found and removed in my own Week-5 model — where `impressions_last_30d` and `impressions_prev_30d` turned out to be the literal ingredients of my own trend label. My question isn't whether this happened here (I can't see the feature list closely enough to know), but whether the write-up could name explicitly which features were checked against the label's own construction, the way a data contract would. Separately: with 57 brands and an 80/20 holdout, was the split grouped by brand? My own model's ROC-AUC moved by nearly 0.2 points once I stopped letting the same client appear in both train and validation (Section 2 below) — a plain random split with many pages per brand risks the same inflation here.


## 2. My model under an honest split (before/after)

Re-running my Week-5 Random Forest model two ways on the exact same data and features: a naive random 80/20 split (ignoring which client each page belongs to), versus the grouped-by-client split I used in Week 5. Same model, same features, same target — only the split logic changes.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

RANDOM_SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_opportunity"] = ((df["trend_direction"] == "down") & (df["avg_position"] > 10)).astype(int)

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "clicks_last_30d", "sessions_last_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_age_days", "days_since_last_update",
    "ctr", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]
categorical_features = ["content_type", "main_intent", "competition_level", "freshness_tier", "provider_used", "model_used"]
for c in categorical_features:
    df[c] = df[c].fillna("missing")

def run_split(train_idx, val_idx, label):
    train, val = df.iloc[train_idx], df.iloc[val_idx]
    X_train, y_train = train[numeric_features + categorical_features], train["is_opportunity"]
    X_val, y_val = val[numeric_features + categorical_features], val["is_opportunity"]
    overlap = set(train["client_id"]) & set(val["client_id"])

    preprocess = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ])
    rf = Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_SEED, n_jobs=-1))])
    rf.fit(X_train, y_train)
    probs = rf.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, probs)
    print(f"{label:28s} | client overlap train/val: {len(overlap):3d} of 32 clients | ROC-AUC: {auc:.3f}")
    return auc

print("BEFORE: naive random 80/20 split (no grouping)")
tr_idx, va_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_SEED, stratify=df["is_opportunity"])
auc_before = run_split(tr_idx, va_idx, "Random split")

print()
print("AFTER: grouped by client_id (honest)")
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx2, va_idx2 = next(gss.split(df, groups=df["client_id"]))
auc_after = run_split(tr_idx2, va_idx2, "Grouped-by-client split")

print()
print(f"Inflation from ignoring client grouping: {auc_before - auc_after:+.3f} ROC-AUC points")


BEFORE: naive random 80/20 split (no grouping)


Random split                 | client overlap train/val:  31 of 32 clients | ROC-AUC: 0.767

AFTER: grouped by client_id (honest)


Grouped-by-client split      | client overlap train/val:   0 of 32 clients | ROC-AUC: 0.573

Inflation from ignoring client grouping: +0.194 ROC-AUC points


**Reading this honestly:** the naive random split let 31 of 32 clients appear in *both* train and validation — meaning the model had already seen pages from almost every client before being "tested" on them. That inflated ROC-AUC by nearly 0.2 points (0.767 vs. the honest 0.573). This is the same category of mistake I asked about in Finding B above — it's a concrete demonstration of why the question matters, not just a hypothetical.


## 3. Leakage audit

Re-checking my final feature set, the same way I'd want my own claims checked.


In [2]:
# Systematic re-check of every feature actually used in the Week-5 / Week-6 model
audit = {
    "trend_direction / trend_pct": "EXCLUDED - literal label source",
    "avg_position / position_tier": "EXCLUDED - avg_position>10 is half the label's definition",
    "impressions_last_30d / impressions_prev_30d": "EXCLUDED - confirmed via data dictionary these are the exact inputs to trend_direction's 30d-vs-prev-30d formula",
    "clicks_last_30d / clicks_prev_30d": "KEPT - not part of the trend_direction formula (that formula uses impressions only, per data dictionary); genuine behavioral signal",
    "sessions_last_30d / sessions_prev_30d": "KEPT - same reasoning as clicks above",
    "impressions_90d / clicks_90d / ctr": "KEPT - a trailing 90-day aggregate, not the same window used to construct the label",
    "content_age_days / days_since_last_update": "KEPT - available at decision time, not label-derived",
    "search_volume / competition / cpc": "KEPT - external keyword-level metadata, unrelated to this page's own performance history",
}
for feature, verdict in audit.items():
    print(f"{feature:55s} -> {verdict}")


trend_direction / trend_pct                             -> EXCLUDED - literal label source
avg_position / position_tier                            -> EXCLUDED - avg_position>10 is half the label's definition
impressions_last_30d / impressions_prev_30d             -> EXCLUDED - confirmed via data dictionary these are the exact inputs to trend_direction's 30d-vs-prev-30d formula
clicks_last_30d / clicks_prev_30d                       -> KEPT - not part of the trend_direction formula (that formula uses impressions only, per data dictionary); genuine behavioral signal
sessions_last_30d / sessions_prev_30d                   -> KEPT - same reasoning as clicks above
impressions_90d / clicks_90d / ctr                      -> KEPT - a trailing 90-day aggregate, not the same window used to construct the label
content_age_days / days_since_last_update               -> KEPT - available at decision time, not label-derived
search_volume / competition / cpc                       -> KEPT - external ke

**What's still a soft spot, honestly:** `clicks_last_30d`/`clicks_prev_30d` and `sessions_last_30d`/`sessions_prev_30d` are correlated with `impressions_last_30d`/`impressions_prev_30d` (the actual excluded label-source columns) even though they aren't literally used in the trend formula. That correlation means some of the same information could be leaking back in indirectly, just not as cleanly as a direct label copy. I'm keeping them because the data dictionary confirms they're not the literal formula inputs, but this is exactly the kind of "soft leakage" that's worth flagging rather than declaring fully clean.


## 4. Claim rewrite

**My boldest sentence, as originally written (from my Week-5 case study and CV):**
> "Random Forest winning on Precision@50."

This states a comparison result as a settled fact, with no scope on which data, which split, or whether it would hold up again.

**Rewritten in safe language:**
> "On one client-grouped validation split, Random Forest showed a higher observed Precision@50 (0.320) than Logistic Regression (0.260), despite a lower overall ROC-AUC. This is a single-split, decision-support observation, not a guaranteed ranking across all future data — a different random seed or a larger validation set could move these numbers."


In [3]:
# No computation needed for this section - documenting the rewrite above is the deliverable.
print("Claim rewrite documented above.")


Claim rewrite documented above.


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are the repo's own pseudonyms)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit my repo URL on the card. Done.
